In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import time
from scipy.stats import ks_2samp, chi2_contingency
from sklearn.base import clone


from src.config import MLFLOW_TRACKING_URI_LOCAL, PIPELINE_SCHEMA_VERSION
from src.features.build_features import (
    drop_features,
    add_new_features,
    make_dummies,
)
import mlflow
import requests

from src.data.make_split import make_split
from src.models.train_model import verify_model

from pgmpy.estimators import HillClimbSearch, BayesianEstimator
from pgmpy.models import DiscreteBayesianNetwork

df_raw = pd.read_csv(
    Path.cwd().parent / "data" / "raw" / "IBM_HR_attr.csv",
    index_col="Unnamed: 0",
)


def preproc(df):
    df = add_new_features(df)
    df = make_dummies(df)
    df = drop_features(df)
    return df


def idiotic_func(x):
    return 0 if x == "No" else 1

e:\miniconda\envs\mlops\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\miniconda\envs\mlops\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [3]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI_LOCAL)

In [9]:
run = mlflow.get_run("539506ca4ae941c7b539b1a3f3f6d2aa")
exp_id = run.info.experiment_id
exp = mlflow.get_experiment(exp_id)
exp_name = exp.name
print(exp_name)

SVM_balanced_2


In [24]:
columns_to_drop = [
    "MonthlyIncome",
    "TotalWorkingYears",
    "YearsInCurrentRole",
    "YearsWithCurrManager",
    "PercentSalaryHike",
    "Over18",
    "EmployeeCount",
    "StandardHours",
    "EmployeeNumber",
    "Attrition_No",
]

all_columns = sorted(list(df_raw.columns))
print(
    "=" * 10,
    "all columns (",
    len(all_columns),
    ") ",
    "=" * 10,
)
print(*all_columns, sep=" ")

usefull_columns = sorted(list(set(all_columns).difference(columns_to_drop)))
real_columns_to_drop = sorted(
    list(set(columns_to_drop).intersection(all_columns))
)

print(
    "=" * 10,
    "usefull_columns (",
    len(usefull_columns),
    ") ",
    "=" * 10,
)
print(*usefull_columns, sep=" ")

print(
    "=" * 10,
    "real_columns_to_drop (",
    len(real_columns_to_drop),
    ") ",
    "=" * 10,
)
print(*real_columns_to_drop, sep=" ")

========== all columns ( 35 )  ==========
Age Attrition BusinessTravel DailyRate Department DistanceFromHome Education EducationField EmployeeCount EmployeeNumber EnvironmentSatisfaction Gender HourlyRate JobInvolvement JobLevel JobRole JobSatisfaction MaritalStatus MonthlyIncome MonthlyRate NumCompaniesWorked Over18 OverTime PercentSalaryHike PerformanceRating RelationshipSatisfaction StandardHours StockOptionLevel TotalWorkingYears TrainingTimesLastYear WorkLifeBalance YearsAtCompany YearsInCurrentRole YearsSinceLastPromotion YearsWithCurrManager
========== usefull_columns ( 26 )  ==========
Age Attrition BusinessTravel DailyRate Department DistanceFromHome Education EducationField EnvironmentSatisfaction Gender HourlyRate JobInvolvement JobLevel JobRole JobSatisfaction MaritalStatus MonthlyRate NumCompaniesWorked OverTime PerformanceRating RelationshipSatisfaction StockOptionLevel TrainingTimesLastYear WorkLifeBalance YearsAtCompany YearsSinceLastPromotion
========== real_columns_to

In [25]:
print(len(all_columns))

35


In [26]:
for jj, col in enumerate(all_columns):
    print(f'{str(col).lower()} AS "{col}",')

age AS "Age",
attrition AS "Attrition",
businesstravel AS "BusinessTravel",
dailyrate AS "DailyRate",
department AS "Department",
distancefromhome AS "DistanceFromHome",
education AS "Education",
educationfield AS "EducationField",
employeecount AS "EmployeeCount",
employeenumber AS "EmployeeNumber",
environmentsatisfaction AS "EnvironmentSatisfaction",
gender AS "Gender",
hourlyrate AS "HourlyRate",
jobinvolvement AS "JobInvolvement",
joblevel AS "JobLevel",
jobrole AS "JobRole",
jobsatisfaction AS "JobSatisfaction",
maritalstatus AS "MaritalStatus",
monthlyincome AS "MonthlyIncome",
monthlyrate AS "MonthlyRate",
numcompaniesworked AS "NumCompaniesWorked",
over18 AS "Over18",
overtime AS "OverTime",
percentsalaryhike AS "PercentSalaryHike",
performancerating AS "PerformanceRating",
relationshipsatisfaction AS "RelationshipSatisfaction",
standardhours AS "StandardHours",
stockoptionlevel AS "StockOptionLevel",
totalworkingyears AS "TotalWorkingYears",
trainingtimeslastyear AS "Training

In [27]:
droping_cols = df_raw.loc[:, real_columns_to_drop]
udf = df_raw.loc[:, usefull_columns]

In [28]:
for jj, col in enumerate(droping_cols.columns):
    un = droping_cols[col].unique()
    unl = len(un)
    print(
        f"{jj + 1}) '{col}' has {unl} unique values ({droping_cols[col].dtype}): ",
        end="",
    )
    if unl > 20:
        print("<too many to show>")
    else:
        print(un)

1) 'EmployeeCount' has 1 unique values (int64): [1]
2) 'EmployeeNumber' has 1470 unique values (int64): <too many to show>
3) 'MonthlyIncome' has 1349 unique values (int64): <too many to show>
4) 'Over18' has 1 unique values (object): ['Y']
5) 'PercentSalaryHike' has 15 unique values (int64): [11 23 15 12 13 20 22 21 17 14 16 18 19 24 25]
6) 'StandardHours' has 1 unique values (int64): [80]
7) 'TotalWorkingYears' has 40 unique values (int64): <too many to show>
8) 'YearsInCurrentRole' has 19 unique values (int64): [ 4  7  0  2  5  9  8  3  6 13  1 15 14 16 11 10 12 18 17]
9) 'YearsWithCurrManager' has 18 unique values (int64): [ 5  7  0  2  6  8  3 11 17  1  4 12  9 10 15 13 16 14]


In [29]:
for jj, col in enumerate(udf.columns):
    un = udf[col].unique()
    unl = len(un)
    print(
        f"{jj + 1}) '{col}' has {unl} unique values ({udf[col].dtype}): ",
        end="",
    )
    if unl > 20:
        print("<too many to show>")
    else:
        print(un)

1) 'Age' has 43 unique values (int64): <too many to show>
2) 'Attrition' has 2 unique values (object): ['Yes' 'No']
3) 'BusinessTravel' has 3 unique values (object): ['Travel_Rarely' 'Travel_Frequently' 'Non-Travel']
4) 'DailyRate' has 886 unique values (int64): <too many to show>
5) 'Department' has 3 unique values (object): ['Sales' 'Research & Development' 'Human Resources']
6) 'DistanceFromHome' has 29 unique values (int64): <too many to show>
7) 'Education' has 5 unique values (int64): [2 1 4 3 5]
8) 'EducationField' has 6 unique values (object): ['Life Sciences' 'Other' 'Medical' 'Marketing' 'Technical Degree'
 'Human Resources']
9) 'EnvironmentSatisfaction' has 4 unique values (int64): [2 3 4 1]
10) 'Gender' has 2 unique values (object): ['Female' 'Male']
11) 'HourlyRate' has 71 unique values (int64): <too many to show>
12) 'JobInvolvement' has 4 unique values (int64): [3 2 4 1]
13) 'JobLevel' has 5 unique values (int64): [2 1 3 4 5]
14) 'JobRole' has 9 unique values (object): [

In [30]:
def make_bn_training_frame(
    df: pd.DataFrame,
    columns: list[str] = usefull_columns,
    max_numeric_states: int = 10,
    n_bins: int = 5,
) -> tuple[pd.DataFrame, dict]:
    """
    Преобразует исходный DataFrame в полностью дискретный DataFrame для Bayesian Network.

    Числовые признаки с большим числом уникальных значений бьются на quantile-bins.
    Числовые признаки с малым числом уникальных значений остаются категориями.
    """

    source = df[columns].copy()

    bn_df = pd.DataFrame(index=source.index)

    meta = {
        "columns": columns,
        "high_card_numeric": {},
        "low_card_numeric": [],
        "categorical": [],
    }

    for col in columns:
        s = source[col]

        if pd.api.types.is_numeric_dtype(s):
            nunique = s.nunique(dropna=True)

            if nunique > max_numeric_states:
                q = min(n_bins, nunique)

                codes = pd.qcut(
                    s,
                    q=q,
                    labels=False,
                    duplicates="drop",
                )

                bn_df[col] = codes.astype("Int64").astype(str)

                values_by_state = {}
                for state in sorted(
                    bn_df[col].dropna().unique(), key=lambda x: int(x)
                ):
                    mask = bn_df[col] == state
                    values_by_state[state] = (
                        s[mask].dropna().astype(int).to_numpy()
                    )

                meta["high_card_numeric"][col] = {
                    "values_by_state": values_by_state,
                    "fallback_values": s.dropna().astype(int).to_numpy(),
                }

            else:
                bn_df[col] = s.astype("Int64").astype(str)
                meta["low_card_numeric"].append(col)

        else:
            bn_df[col] = s.astype(str)
            meta["categorical"].append(col)

    return bn_df, meta


def fit_bayesian_generator(
    df: pd.DataFrame,
    max_indegree: int = 3,
    max_iter: int = 2_000,
    n_bins: int = 5,
    equivalent_sample_size: int = 5,
    scoring_method="bic-d",
):
    """
    1. Дискретизирует данные.
    2. Учит структуру Bayesian Network.
    3. Оценивает CPD.
    """

    bn_df, meta = make_bn_training_frame(
        df=df,
        n_bins=n_bins,
    )

    start = time.perf_counter()

    structure_estimator = HillClimbSearch(data=bn_df)

    dag = structure_estimator.estimate(
        scoring_method=scoring_method,
        max_indegree=max_indegree,
        max_iter=max_iter,
        show_progress=True,
    )

    model = DiscreteBayesianNetwork(ebunch=list(dag.edges()))
    model.add_nodes_from(bn_df.columns)

    estimator = BayesianEstimator(model, bn_df)

    cpds = estimator.get_parameters(
        prior_type="BDeu",
        equivalent_sample_size=equivalent_sample_size,
        n_jobs=1,
    )

    model.add_cpds(*cpds)
    model.check_model()

    fit_seconds = time.perf_counter() - start

    return model, meta, fit_seconds


def decode_bn_samples(
    samples_bn: pd.DataFrame,
    meta: dict,
    seed: int | None = None,
) -> pd.DataFrame:
    """
    Возвращает сгенерированные BN данные обратно в формат, похожий на исходный:
    bins -> реальные int значения из соответствующих bin-ов.
    """

    rng = np.random.default_rng(seed)
    result = pd.DataFrame(index=samples_bn.index)

    for col in meta["columns"]:
        if col in meta["high_card_numeric"]:
            info = meta["high_card_numeric"][col]
            values = []

            for state in samples_bn[col].astype(str):
                pool = info["values_by_state"].get(state)

                if pool is None or len(pool) == 0:
                    pool = info["fallback_values"]

                values.append(int(rng.choice(pool)))

            result[col] = values

        elif col in meta["low_card_numeric"]:
            result[col] = pd.to_numeric(
                samples_bn[col], errors="coerce"
            ).astype(int)

        else:
            result[col] = samples_bn[col].astype(str)

    return result


def generate_synthetic_data(
    model: DiscreteBayesianNetwork,
    meta: dict,
    n_samples: int = 100,
    seed: int | None = 42,
) -> pd.DataFrame:
    """
    Генерирует синтетические данные в исходном формате.
    """

    samples_bn = model.simulate(
        n_samples=n_samples,
        seed=seed,
        show_progress=False,
    )

    return decode_bn_samples(samples_bn, meta, seed=seed)


def make_target_drift(
    df: pd.DataFrame, strength: float = 0.2, seed: int | None = 18
) -> pd.DataFrame:
    """
    Искусственный target drift для демонстрации.
    """

    rng = np.random.default_rng(seed)
    drifted = df.copy()
    n = len(drifted)

    mask = rng.random(n) < strength

    if "Attrition" in drifted.columns:
        drifted.loc[mask, "Attrition"] = rng.choice(
            ["Yes", "No"],
            size=mask.sum(),
            p=[0.75, 0.25],
        )


def make_data_drift(
    df: pd.DataFrame,
    strength: float = 0.5,
    seed: int | None = 18,
) -> pd.DataFrame:
    """
    Искусственный data drift для демонстрации.

    Меняем распределения входных признаков:
    - больше OverTime = Yes
    - больше Travel_Frequently
    - ниже EnvironmentSatisfaction
    - выше DistanceFromHome
    - больше Single
    """

    rng = np.random.default_rng(seed)
    drifted = df.copy()
    n = len(drifted)

    mask = rng.random(n) < strength

    if "OverTime" in drifted.columns:
        drifted.loc[mask, "OverTime"] = rng.choice(
            ["Yes", "No"],
            size=mask.sum(),
            p=[0.75, 0.25],
        )

    if "BusinessTravel" in drifted.columns:
        drifted.loc[mask, "BusinessTravel"] = rng.choice(
            ["Travel_Frequently", "Travel_Rarely", "Non-Travel"],
            size=mask.sum(),
            p=[0.65, 0.30, 0.05],
        )

    if "EnvironmentSatisfaction" in drifted.columns:
        drifted.loc[mask, "EnvironmentSatisfaction"] = rng.choice(
            [1, 2, 3, 4],
            size=mask.sum(),
            p=[0.45, 0.35, 0.15, 0.05],
        )

    if "DistanceFromHome" in drifted.columns:
        drifted.loc[mask, "DistanceFromHome"] = rng.choice(
            np.arange(15, 30),
            size=mask.sum(),
        )

    if "MaritalStatus" in drifted.columns:
        drifted.loc[mask, "MaritalStatus"] = rng.choice(
            ["Single", "Married", "Divorced"],
            size=mask.sum(),
            p=[0.65, 0.25, 0.10],
        )

    return drifted


def make_data_drift_low_impact(
    df: pd.DataFrame,
    strength: float = 0.5,
    seed: int | None = 42,
) -> pd.DataFrame:
    """
    искусственный data drift по признакам,
    которые слабо влияют на Attrition.

    """

    rng = np.random.default_rng(seed)
    drifted = df.copy()
    n = len(drifted)

    mask = rng.random(n) < strength
    n_changed = mask.sum()

    if n_changed == 0:
        return drifted

    # 1. DailyRate: сдвигаем распределение в сторону больших значений
    if "DailyRate" in drifted.columns:
        values = drifted["DailyRate"].dropna().to_numpy()
        high_values = values[values >= np.quantile(values, 0.7)]

        if len(high_values) > 0:
            drifted.loc[mask, "DailyRate"] = rng.choice(
                high_values,
                size=n_changed,
                replace=True,
            )

    # 2. HourlyRate: сдвигаем распределение в сторону меньших значений
    if "HourlyRate" in drifted.columns:
        values = drifted["HourlyRate"].dropna().to_numpy()
        low_values = values[values <= np.quantile(values, 0.3)]

        if len(low_values) > 0:
            drifted.loc[mask, "HourlyRate"] = rng.choice(
                low_values,
                size=n_changed,
                replace=True,
            )

    # 3. MonthlyRate: делаем сдвиг в верхнюю часть распределения
    if "MonthlyRate" in drifted.columns:
        values = drifted["MonthlyRate"].dropna().to_numpy()
        high_values = values[values >= np.quantile(values, 0.75)]

        if len(high_values) > 0:
            drifted.loc[mask, "MonthlyRate"] = rng.choice(
                high_values,
                size=n_changed,
                replace=True,
            )

    # 4. TrainingTimesLastYear: немного меняем распределение,
    # но не уводим в экстремальные значения
    if "TrainingTimesLastYear" in drifted.columns:
        drifted.loc[mask, "TrainingTimesLastYear"] = rng.choice(
            [1, 2, 3, 4],
            size=n_changed,
            p=[0.15, 0.25, 0.45, 0.15],
        )

    # 5. PerformanceRating: чаще ставим 3, реже 4
    # Обычно этот признак почти константный и малоинформативный
    if "PerformanceRating" in drifted.columns:
        drifted.loc[mask, "PerformanceRating"] = rng.choice(
            [3, 4],
            size=n_changed,
            p=[0.95, 0.05],
        )

    return drifted

In [31]:
FEATURE_COLUMNS = [
    "Age",
    "BusinessTravel",
    "DailyRate",
    "Department",
    "DistanceFromHome",
    "Education",
    "EducationField",
    "EnvironmentSatisfaction",
    "Gender",
    "HourlyRate",
    "JobInvolvement",
    "JobLevel",
    "JobRole",
    "JobSatisfaction",
    "MaritalStatus",
    "MonthlyRate",
    "NumCompaniesWorked",
    "OverTime",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "TrainingTimesLastYear",
    "WorkLifeBalance",
    "YearsAtCompany",
    "YearsSinceLastPromotion",
]


CATEGORICAL_COLUMNS = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "Gender",
    "JobRole",
    "MaritalStatus",
    "OverTime",
]


def compare_feature_drift(
    real_df: pd.DataFrame,
    synthetic_df: pd.DataFrame,
    feature_columns: list[str] = FEATURE_COLUMNS,
    categorical_columns: list[str] = CATEGORICAL_COLUMNS,
    alpha: float = 0.05,
    min_expected_count: int = 5,
) -> pd.DataFrame:
    """
    Сравнивает распределения признаков в real_df и synthetic_df.

    Для числовых признаков:
        KS-test.

    Для категориальных признаков:
        Chi-square test.

    Возвращает DataFrame с p-value и флагом drift_detected.
    """

    results = []

    categorical_columns = set(categorical_columns)

    for col in feature_columns:
        if col not in real_df.columns or col not in synthetic_df.columns:
            results.append(
                {
                    "feature": col,
                    "test": "missing_column",
                    "statistic": np.nan,
                    "p_value": np.nan,
                    "drift_detected": None,
                    "note": "column missing in one of dataframes",
                }
            )
            continue

        real_s = real_df[col].dropna()
        synth_s = synthetic_df[col].dropna()

        if len(real_s) == 0 or len(synth_s) == 0:
            results.append(
                {
                    "feature": col,
                    "test": "empty_column",
                    "statistic": np.nan,
                    "p_value": np.nan,
                    "drift_detected": None,
                    "note": "empty after dropna",
                }
            )
            continue

        if col in categorical_columns:
            all_categories = sorted(
                set(real_s.astype(str).unique())
                | set(synth_s.astype(str).unique())
            )

            real_counts = (
                real_s.astype(str)
                .value_counts()
                .reindex(all_categories, fill_value=0)
            )
            synth_counts = (
                synth_s.astype(str)
                .value_counts()
                .reindex(all_categories, fill_value=0)
            )

            contingency = np.vstack([real_counts.values, synth_counts.values])

            try:
                stat, p_value, _, expected = chi2_contingency(contingency)

                note = ""
                if (expected < min_expected_count).any():
                    note = "some expected counts are low; chi-square may be unstable"

                results.append(
                    {
                        "feature": col,
                        "test": "chi2",
                        "statistic": stat,
                        "p_value": p_value,
                        "drift_detected": bool(p_value < alpha),
                        "note": note,
                    }
                )

            except ValueError as e:
                results.append(
                    {
                        "feature": col,
                        "test": "chi2",
                        "statistic": np.nan,
                        "p_value": np.nan,
                        "drift_detected": None,
                        "note": str(e),
                    }
                )

        else:
            real_num = pd.to_numeric(real_s, errors="coerce").dropna()
            synth_num = pd.to_numeric(synth_s, errors="coerce").dropna()

            if len(real_num) == 0 or len(synth_num) == 0:
                results.append(
                    {
                        "feature": col,
                        "test": "ks",
                        "statistic": np.nan,
                        "p_value": np.nan,
                        "drift_detected": None,
                        "note": "non-numeric values after conversion",
                    }
                )
                continue

            stat, p_value = ks_2samp(real_num, synth_num)

            results.append(
                {
                    "feature": col,
                    "test": "ks",
                    "statistic": stat,
                    "p_value": p_value,
                    "drift_detected": bool(p_value < alpha),
                    "note": "",
                }
            )

    result_df = pd.DataFrame(results)

    result_df = result_df.sort_values(
        by=["drift_detected", "p_value"],
        ascending=[False, True],
        na_position="last",
    ).reset_index(drop=True)

    return result_df

In [32]:
bn_model, bn_meta, fit_seconds = fit_bayesian_generator(
    udf, max_indegree=5, n_bins=20, max_iter=2_000
)

print(f"Bayesian Network fitted in {fit_seconds:.2f} seconds")
print("Edges:", list(bn_model.edges())[:20])

C:\Users\Mishas\AppData\Local\Temp\ipykernel_19256\1468985358.py:89: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  structure_estimator = HillClimbSearch(data=bn_df)
  1%|          | 12/2000 [00:03<09:41,  3.42it/s] 
C:\Users\Mishas\AppData\Local\Temp\ipykernel_19256\1468985358.py:101: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(model, bn_df)


Bayesian Network fitted in 3.79 seconds
Edges: [('Attrition', 'OverTime'), ('Attrition', 'MaritalStatus'), ('Attrition', 'BusinessTravel'), ('Attrition', 'JobInvolvement'), ('MaritalStatus', 'StockOptionLevel'), ('Department', 'EducationField'), ('JobLevel', 'JobRole'), ('JobLevel', 'Age'), ('JobLevel', 'Attrition'), ('JobRole', 'Department'), ('YearsAtCompany', 'YearsSinceLastPromotion'), ('YearsAtCompany', 'JobLevel')]


In [33]:
try:
    requests.get(MLFLOW_TRACKING_URI_LOCAL, timeout=10.0)
except requests.RequestException:
    print("MLflow server unavailible, ", end="")
    print("predictorClass tests are skipped")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI_LOCAL)
runs = mlflow.search_runs(
    search_all_experiments=True,
    filter_string=(
        "tags.pipeline_schema_version = "
        f"'{PIPELINE_SCHEMA_VERSION}' AND tags.model_saved = 'True'"
    ),
)

if len(runs) == 0:
    print("There are no suitable models in mlflow, ", end="")
    print("predictorClass tests are skipped")
else:
    run_id = runs.iloc[0]["run_id"]
    model_uri = f"runs:/{run_id}/model"
    model = mlflow.sklearn.load_model(model_uri)

MLflow server unavailible, predictorClass tests are skipped


KeyboardInterrupt: 

## Пропускаем далее

In [329]:
1 / 0

ZeroDivisionError: division by zero

In [34]:
df_syn = generate_synthetic_data(
    model=bn_model,
    meta=bn_meta,
    n_samples=1470,  # same as df len part
    seed=123,
)

for col in droping_cols.columns:
    df_syn[col] = droping_cols[col]

In [35]:
df_raw_y = df_raw["Attrition"].copy().apply(idiotic_func)
df_syn_y = df_syn["Attrition"].copy().apply(idiotic_func)

raw_x = preproc(df_raw)
syn_x = preproc(df_syn)

raw_results = verify_model(model, raw_x, df_raw_y)
print("===" * 10)
print(raw_results)

NameError: name 'model' is not defined

In [36]:
syn_results = verify_model(model, syn_x, df_syn_y)
print("===" * 10)
print(syn_results)

NameError: name 'model' is not defined

In [37]:
drift_report = compare_feature_drift(
    real_df=df_raw,
    synthetic_df=df_syn,
    alpha=0.05,
)

drift_report

e:\miniconda\envs\mlops\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


,feature,test,statistic,p_value,drift_detected,note
0,OverTime,chi2,1.677309,0.195282,False,
1,NumCompaniesWorked,ks,0.031973,0.440270,False,
2,DailyRate,ks,0.029932,0.525727,False,
3,Department,chi2,0.959586,0.618911,False,
4,EducationField,chi2,3.511247,0.621687,False,
5,TrainingTimesLastYear,ks,0.027211,0.648055,False,
6,Gender,chi2,0.114477,0.735104,False,
7,MonthlyRate,ks,0.025170,0.740512,False,
8,JobRole,chi2,5.130630,0.743527,False,
9,MaritalStatus,chi2,0.541287,0.762889,False,


In [41]:
bool(drift_report["drift_detected"].max())

False

In [42]:
drift_report = compare_feature_drift(
    real_df=df_raw,
    synthetic_df=df_syn,
    alpha=0.05,
    feature_columns=["Attrition"],
    categorical_columns=["Attrition"],
)

drift_report

,feature,test,statistic,p_value,drift_detected,note
0,Attrition,chi2,1.069555,0.301046,False,


In [45]:
bool(drift_report.iloc[0]["drift_detected"])

False

In [ ]:
raw_predictions = pd.DataFrame(
    data=model.predict(raw_x), columns=["Attrition"]
)
syn_predictions = pd.DataFrame(
    data=model.predict(syn_x), columns=["Attrition"]
)

In [ ]:
drift_report = compare_feature_drift(
    real_df=raw_predictions,
    synthetic_df=syn_predictions,
    alpha=0.05,
    feature_columns=["Attrition"],
    categorical_columns=["Attrition"],
)

drift_report

,feature,test,statistic,p_value,drift_detected,note
0,Attrition,chi2,8.845334,0.002938,True,


In [ ]:
df_syng = df_syn.copy()

print("Prediction:")
print("raw mean:", raw_predictions.mean().item())
print("syn mean:", syn_predictions.mean().item())

Prediction:
raw mean: 0.30680272108843537
syn mean: 0.35918367346938773


In [ ]:
df_raw["OverTime"].describe()

count     1470
unique       2
top         No
freq      1054
Name: OverTime, dtype: object

In [ ]:
overtime_no_ratio = 1054 / 1470
overtime_yes_ratio = 1.0 - overtime_no_ratio

print("No:", overtime_no_ratio)
print("Yes:", overtime_yes_ratio)

No: 0.7170068027210884
Yes: 0.2829931972789116


In [ ]:
coef = 1.066

df_syn2 = generate_synthetic_data(
    model=bn_model,
    meta=bn_meta,
    n_samples=int(1470 * coef),  # same as df len part
    seed=None,
)

added_rows = int(1470 * coef) - 1470

for col in droping_cols.columns:
    temp = droping_cols[col]
    df_syn2[col] = pd.concat(
        [temp, temp.iloc[0:added_rows]], ignore_index=True
    )

print(len(df_syn2))

syn2_x = preproc(df_syn2)
syn2_y = df_syn2["Attrition"].copy().apply(idiotic_func)

syn2_p = model.predict(syn2_x)

p_raw_mean = raw_predictions.mean().item()
p_syn2_exp_mean = syn2_p.mean().item()

print("Prediction:")
print("raw mean:", p_raw_mean)
print("syn2 mean:", p_syn2_exp_mean)

1567
Prediction:
raw mean: 0.30680272108843537
syn2 mean: 0.34652201659221443


In [ ]:
df_syn2_exp = df_syn2.copy()
df_syn2_exp["pred"] = syn2_p
df_syn2_exp["attr_dummie"] = syn2_y

num_not_attr = added_rows // 2
num_attr = added_rows - num_not_attr

num_not_attr_overtime_no = int(num_not_attr * overtime_no_ratio)
num_not_attr_overtime_yes = num_not_attr - num_not_attr_overtime_no

num_attr_overtime_no = int(num_attr * overtime_no_ratio)
num_attr_overtime_yes = num_attr - num_attr_overtime_no


gt_non_attr_over_time_no = (
    df_syn2_exp[
        (df_syn2_exp.pred == 1)
        & (df_syn2_exp.attr_dummie == 0)
        & (df_syn2_exp.OverTime == "No")
    ]
    .sample(num_not_attr_overtime_no)
    .index
)

gt_non_attr_over_time_yes = (
    df_syn2_exp[
        (df_syn2_exp.pred == 1)
        & (df_syn2_exp.attr_dummie == 0)
        & (df_syn2_exp.OverTime == "Yes")
    ]
    .sample(num_not_attr_overtime_yes)
    .index
)

gt_attr_over_time_no = (
    df_syn2_exp[
        (df_syn2_exp.pred == 1)
        & (df_syn2_exp.attr_dummie == 1)
        & (df_syn2_exp.OverTime == "No")
    ]
    .sample(num_attr_overtime_no)
    .index
)

gt_attr_over_time_yes = (
    df_syn2_exp[
        (df_syn2_exp.pred == 1)
        & (df_syn2_exp.attr_dummie == 1)
        & (df_syn2_exp.OverTime == "Yes")
    ]
    .sample(num_attr_overtime_yes)
    .index
)

print("len(gt_non_attr_over_time_no)", len(gt_non_attr_over_time_no))
print("len(gt_non_attr_over_time_yes)", len(gt_non_attr_over_time_yes))
print("len(gt_attr_over_time_no)", len(gt_attr_over_time_no))
print("len(gt_attr_over_time_yes)", len(gt_attr_over_time_yes))

df_syn2_exp.drop(index=gt_non_attr_over_time_no, inplace=True)
df_syn2_exp.drop(index=gt_non_attr_over_time_yes, inplace=True)
df_syn2_exp.drop(index=gt_attr_over_time_no, inplace=True)
df_syn2_exp.drop(index=gt_attr_over_time_yes, inplace=True)
df_syn2_exp.drop(columns=["pred", "attr_dummie"])


syn2_exp_x = preproc(df_syn2_exp)
syn2_exp_y = df_syn2["Attrition"].copy().apply(idiotic_func)
syn2_exp_p = model.predict(syn2_exp_x)

print()
print("Gt attr:")
print("raw mean:", df_raw_y.mean().item())
print("syn2_exp mean:", syn2_exp_y.mean().item())

print()
print("Prediction:")
print("raw mean:", raw_predictions.mean().item())
print("syn2_exp mean:", syn2_exp_p.mean().item())

len(gt_non_attr_over_time_no) 34
len(gt_non_attr_over_time_yes) 14
len(gt_attr_over_time_no) 35
len(gt_attr_over_time_yes) 14

Gt attr:
raw mean: 0.16122448979591836
syn2_exp mean: 0.16783663050414804

Prediction:
raw mean: 0.30680272108843537
syn2_exp mean: 0.3034013605442177


In [ ]:
drift_report = compare_feature_drift(
    real_df=df_raw,
    synthetic_df=df_syn2_exp,
    alpha=0.05,
)

drift_report

,feature,test,statistic,p_value,drift_detected,note
0,MaritalStatus,chi2,3.219323,0.199955,False,
1,OverTime,chi2,1.575178,0.209457,False,
2,HourlyRate,ks,0.031973,0.440270,False,
3,StockOptionLevel,ks,0.028571,0.586133,False,
4,Age,ks,0.027891,0.616987,False,
5,JobLevel,ks,0.026531,0.679146,False,
6,BusinessTravel,chi2,0.703188,0.703566,False,
7,YearsAtCompany,ks,0.025170,0.740512,False,
8,Education,ks,0.024490,0.770298,False,
9,EducationField,chi2,2.375874,0.795061,False,


In [ ]:
syn2_exp_predictions = pd.DataFrame(data=syn2_exp_p, columns=["Attrition"])

drift_report = compare_feature_drift(
    real_df=raw_predictions,
    synthetic_df=syn2_exp_predictions,
    alpha=0.05,
    feature_columns=["Attrition"],
    categorical_columns=["Attrition"],
)

drift_report

,feature,test,statistic,p_value,drift_detected,note
0,Attrition,chi2,0.025669,0.872712,False,


In [ ]:
syn2_exp_predictions = pd.DataFrame(data=syn2_exp_p, columns=["Attrition"])

drift_report = compare_feature_drift(
    real_df=df_raw,
    synthetic_df=df_syn2_exp,
    alpha=0.05,
    feature_columns=["Attrition"],
    categorical_columns=["Attrition"],
)

drift_report

,feature,test,statistic,p_value,drift_detected,note
0,Attrition,chi2,1.267626,0.260212,False,


In [ ]:
raw_results = verify_model(model, raw_x, df_raw_y)
print("===" * 10)
print(raw_results)

Оптимальный порог по f1-score: 1.0000
F1 Score: 0.5406976744186046
ROC AUC: 0.7849435872165245
PR AUC: 0.35836279893528605
Confusion matrix: 
 [[968 265]
 [ 51 186]]
{'f1_Score': 0.5406976744186046, 'precision': 0.4124168514412417, 'recall': 0.7848101265822784, 'roc_auc': 0.7849435872165245, 'pr_auc': 0.35836279893528605}


In [ ]:
df_syn2_exp_c = df_syn2_exp.copy()

syn2_exp_c_x = preproc(df_syn2_exp_c)
syn2_exp_c_y = df_syn2_exp_c["Attrition"].copy().apply(idiotic_func)


syn2_exp_c_results = verify_model(model, syn2_exp_c_x, syn2_exp_c_y)
print("===" * 10)
print(syn_results)

Оптимальный порог по f1-score: 1.0000
F1 Score: 0.27575757575757576
ROC AUC: 0.5712951663789511
PR AUC: 0.17043638795780666
Confusion matrix: 
 [[901 355]
 [123  91]]
{'f1_Score': 0.36086404066073696, 'precision': 0.2689393939393939, 'recall': 0.5482625482625483, 'roc_auc': 0.6147588546432478, 'pr_auc': 0.22704123418409133}


In [ ]:
df_syn2_exp.to_csv(Path.cwd().parent / "data" / "interim" / "syn1.csv")

1470

### Готовим запросы

In [339]:
def add_useles_rows(df_in):
    df_in_len = len(df_in)
    base_col_len = len(droping_cols)
    mult = df_in_len // base_col_len
    rr = df_in_len - mult * base_col_len
    for col in droping_cols.columns:

        temp = [droping_cols[col] for _ in range(mult)]
        if rr > 0:
            temp.append(droping_cols[col].iloc[0:rr])

        df_in[col] = pd.concat(temp, ignore_index=True)
    return df_in


def drift_detect(req, ref, model):
    cat_cols = [
        "Attrition",
        "BusinessTravel",
        "Department",
        "EducationField",
        "Gender",
        "JobRole",
        "MaritalStatus",
        "OverTime",
    ]
    drift_report = compare_feature_drift(
        real_df=ref.copy(),
        synthetic_df=req.copy(),
        alpha=0.05,
        feature_columns=usefull_columns,
        categorical_columns=cat_cols,
    )

    display(drift_report[drift_report.drift_detected == True].head(27))

    target_ref = ref["Attrition"].copy().apply(idiotic_func)
    target_req = req["Attrition"].copy().apply(idiotic_func)

    prep_ref = preproc(ref)
    prep_req = preproc(req)

    # pred_ref = pd.DataFrame(data = model.predict(prep_ref), columns=["Attrition"])
    # pred_req = pd.DataFrame(data = model.predict(prep_req), columns=["Attrition"])

    ref_ans = verify_model(model, prep_ref, target_ref, verbose=False)

    req_ans = verify_model(model, prep_req, target_req, verbose=False)

    print("Reference first, request second row")
    for key in ref_ans:
        print(f"{key}={ref_ans[key]:.3f}".ljust(15), end=" ")
    print()
    for key in req_ans:
        print(f"{key}={req_ans[key]:.3f}".ljust(15), end=" ")


part_len = 1470

In [340]:
# 1) повтор обучающей выборки с помешиванием синтетики (никаких дрифтов)
syn = generate_synthetic_data(
    model=bn_model,
    meta=bn_meta,
    n_samples=2 * 530,
    seed=394944,
)

syn = add_useles_rows(syn)

req = pd.concat([df_raw.copy(), syn, df_raw.copy()], ignore_index=True).sample(
    frac=1
)
req = req.sample(part_len * 2)
print("len(req) =", len(req))
drift_detect(req, df_raw, model)

len(req) = 2940


,feature,test,statistic,p_value,drift_detected,note


Reference first, request second row
f1_Score=0.541  precision=0.412 recall=0.785    roc_auc=0.785   pr_auc=0.358    
f1_Score=0.501  precision=0.375 recall=0.752    roc_auc=0.757   pr_auc=0.322    

In [341]:
# сохраняем
req.to_csv(Path.cwd().parent / "data" / "interim" / "no_drift.csv")
del req

In [342]:
# 2) синтетика с концепт дрифтом
req = generate_synthetic_data(
    model=bn_model,
    meta=bn_meta,
    n_samples=part_len,
    seed=831889,
)

req = add_useles_rows(req)

print("len(req) =", len(req))
drift_detect(req, df_raw, model)

len(req) = 1470


,feature,test,statistic,p_value,drift_detected,note


Reference first, request second row
f1_Score=0.541  precision=0.412 recall=0.785    roc_auc=0.785   pr_auc=0.358    
f1_Score=0.385  precision=0.272 recall=0.658    roc_auc=0.674   pr_auc=0.230    

In [343]:
# сохраняем
req.to_csv(Path.cwd().parent / "data" / "interim" / "concept_drift.csv")
concept_drifted_data = req.copy()
del req

In [344]:
def retrain(df: pd.DataFrame, model):
    (x_train, y_train), (x_val, y_val), (x_test, y_test) = make_split(
        preproc(df)
    )
    model2 = clone(model)
    model2.fit(x_train, y_train)
    metrics_retrained = verify_model(model2, x_val, y_val, verbose=False)
    for key in metrics_retrained:
        print(f"{key}={metrics_retrained[key]:.3f}".ljust(15), end=" ")

    return model2

In [345]:
# 5) просто синтетика, не должно появляться какого-либо дрифта на переученной модели

# переобучаем модель
print("retrain results:")
model_conceptdrifted = retrain(concept_drifted_data.copy(), model)
print()

retrain results:
f1_Score=0.424  precision=0.318 recall=0.636    roc_auc=0.698   pr_auc=0.257    


NameError: name 'mlflow' is not defined

In [346]:
n = 1000
x = np.random.randint(999999)
print("seed", x)


req = generate_synthetic_data(
    model=bn_model,
    meta=bn_meta,
    n_samples=n,
    seed=936773,
)


req = add_useles_rows(req)

req = pd.concat([concept_drifted_data, req], ignore_index=True).sample(
    part_len
)

req = pd.concat([req, req, req]).sample(frac=1)

print("len(req) =", len(req))
drift_detect(req.iloc[:part_len], concept_drifted_data, model_conceptdrifted)
print()
print("===" * 10)
drift_detect(
    req.iloc[part_len : 2 * part_len],
    concept_drifted_data,
    model_conceptdrifted,
)
print()
print("===" * 10)
drift_detect(
    req.iloc[2 * part_len :], concept_drifted_data, model_conceptdrifted
)
print()
print("===" * 10)

seed 893525
len(req) = 4410


,feature,test,statistic,p_value,drift_detected,note


Reference first, request second row
f1_Score=0.444  precision=0.326 recall=0.694    roc_auc=0.722   pr_auc=0.272    
f1_Score=0.434  precision=0.321 recall=0.672    roc_auc=0.705   pr_auc=0.267    


,feature,test,statistic,p_value,drift_detected,note


Reference first, request second row
f1_Score=0.444  precision=0.326 recall=0.694    roc_auc=0.722   pr_auc=0.272    
f1_Score=0.456  precision=0.338 recall=0.702    roc_auc=0.740   pr_auc=0.279    


,feature,test,statistic,p_value,drift_detected,note


Reference first, request second row
f1_Score=0.444  precision=0.326 recall=0.694    roc_auc=0.722   pr_auc=0.272    
f1_Score=0.458  precision=0.340 recall=0.700    roc_auc=0.733   pr_auc=0.282    


In [347]:
# сохраняем
req.to_csv(Path.cwd().parent / "data" / "interim" / "synt_1.csv")
del req

In [348]:
# 6) синтетика, c target-drift

attr_yes_ratio = (concept_drifted_data.Attrition == "Yes").mean()
attr_no_ration = (concept_drifted_data.Attrition == "No").mean()

print(attr_yes_ratio)
print(attr_no_ration)

0.1489795918367347
0.8510204081632653


In [349]:
x = np.random.randint(999999)
print("seed", x)

req = generate_synthetic_data(
    model=bn_model,
    meta=bn_meta,
    n_samples=10000,
    seed=489348,
)

req = pd.concat([req, concept_drifted_data], ignore_index=True)

req_yes = req[req.Attrition == "Yes"].sample(294)
req_no = req[req.Attrition == "No"].sample(1176)
req = pd.concat([req_yes, req_no], ignore_index=True)

del req_no, req_yes
req = add_useles_rows(req)

print("len(req) =", len(req))
drift_detect(req, concept_drifted_data, model_conceptdrifted)

seed 59130
len(req) = 1470


,feature,test,statistic,p_value,drift_detected,note
0,Attrition,chi2,12.930747,0.000323,True,


Reference first, request second row
f1_Score=0.444  precision=0.326 recall=0.694    roc_auc=0.722   pr_auc=0.272    
f1_Score=0.481  precision=0.379 recall=0.656    roc_auc=0.694   pr_auc=0.318    

In [350]:
# сохраняем
req.to_csv(Path.cwd().parent / "data" / "interim" / "target_drift.csv")
target_drifted_data = req.copy()
del concept_drifted_data
del req

In [351]:
# 9) синтетика без дрифта после переобучения
# переобучаем модель
print("retrain results:")
model_targetdrifted = retrain(target_drifted_data.copy(), model)
print()

retrain results:
f1_Score=0.488  precision=0.381 recall=0.678    roc_auc=0.701   pr_auc=0.323    


In [357]:
x = np.random.randint(999999)
print("seed", x)

req = generate_synthetic_data(
    model=bn_model,
    meta=bn_meta,
    n_samples=30000,
    seed=297550,
)

req_yes = req[req.Attrition == "Yes"].sample(294 * 3)
req_no = req[req.Attrition == "No"].sample(1176 * 3)
req = pd.concat([req_yes, req_no], ignore_index=True).sample(part_len * 3)

del req_no, req_yes
req = add_useles_rows(req)

print("len(req) =", len(req))
drift_detect(req.iloc[:part_len], target_drifted_data, model_targetdrifted)
print()
print("===" * 20)

drift_detect(
    req.iloc[part_len : 2 * part_len], target_drifted_data, model_targetdrifted
)
print()
print("===" * 20)

drift_detect(
    req.iloc[2 * part_len :], target_drifted_data, model_targetdrifted
)
print()
print("===" * 20)

seed 499657
len(req) = 4410


,feature,test,statistic,p_value,drift_detected,note


Reference first, request second row
f1_Score=0.512  precision=0.401 recall=0.707    roc_auc=0.722   pr_auc=0.342    
f1_Score=0.486  precision=0.384 recall=0.661    roc_auc=0.696   pr_auc=0.323    


e:\miniconda\envs\mlops\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


,feature,test,statistic,p_value,drift_detected,note


Reference first, request second row
f1_Score=0.512  precision=0.401 recall=0.707    roc_auc=0.722   pr_auc=0.342    
f1_Score=0.491  precision=0.379 recall=0.696    roc_auc=0.702   pr_auc=0.325    


,feature,test,statistic,p_value,drift_detected,note


Reference first, request second row
f1_Score=0.512  precision=0.401 recall=0.707    roc_auc=0.722   pr_auc=0.342    
f1_Score=0.470  precision=0.363 recall=0.667    roc_auc=0.693   pr_auc=0.307    


In [358]:
# сохраняем
req.to_csv(Path.cwd().parent / "data" / "interim" / "synt_2.csv")
del req

In [385]:
# 10) datadrift

x = np.random.randint(999999)
print("seed", x)

req = generate_synthetic_data(
    model=bn_model,
    meta=bn_meta,
    n_samples=10000,
    seed=923594,
)

req_yes = req[req.Attrition == "Yes"].sample(294)
req_no = req[req.Attrition == "No"].sample(1176)
req = pd.concat([req_yes, req_no], ignore_index=True)

del req_no, req_yes
req = add_useles_rows(req)

req = make_data_drift_low_impact(df=req, strength=0.5)

print("len(req) =", len(req))
drift_detect(req, target_drifted_data, model_targetdrifted)

seed 464468
len(req) = 1470


,feature,test,statistic,p_value,drift_detected,note
0,HourlyRate,ks,0.374830,2.572693e-92,True,
1,MonthlyRate,ks,0.352381,2.131275e-81,True,
2,DailyRate,ks,0.337415,1.579874e-74,True,
3,TrainingTimesLastYear,ks,0.072789,8.254971e-04,True,
4,BusinessTravel,chi2,6.592746,3.701718e-02,True,
5,PerformanceRating,ks,0.051020,4.355405e-02,True,


Reference first, request second row
f1_Score=0.512  precision=0.401 recall=0.707    roc_auc=0.722   pr_auc=0.342    
f1_Score=0.485  precision=0.379 recall=0.673    roc_auc=0.699   pr_auc=0.321    

In [371]:
# сохраняем
req.to_csv(Path.cwd().parent / "data" / "interim" / "datadrift.csv")
datadrifted_data = req.copy()
del req

In [380]:
## 12) просто синтетика
print("retrain results:")
model_datadrifted = retrain(datadrifted_data.copy(), model)
print()

retrain results:
f1_Score=0.436  precision=0.340 recall=0.610    roc_auc=0.656   pr_auc=0.285    


In [390]:
x = np.random.randint(999999)
print("seed", x)

lll = 3

req = generate_synthetic_data(
    model=bn_model,
    meta=bn_meta,
    n_samples=lll * 10000,
    seed=792847,
)

req_yes = req[req.Attrition == "Yes"].sample(294 * lll)
req_no = req[req.Attrition == "No"].sample(1176 * lll)
req = pd.concat([req_yes, req_no], ignore_index=True)
del req_no, req_yes
req = add_useles_rows(req)
req = make_data_drift_low_impact(df=req, strength=0.5)

temp = [datadrifted_data for _ in range(lll)]
temp.append(req)
req = pd.concat(temp, ignore_index=True).sample(part_len * lll)


print("len(req) =", len(req))
for l in range(lll):
    drift_detect(
        req.iloc[part_len * l : part_len * (l + 1)],
        datadrifted_data,
        model_datadrifted,
    )
    print()
    print("===" * 20)

seed 698573
len(req) = 4410


e:\miniconda\envs\mlops\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


,feature,test,statistic,p_value,drift_detected,note


Reference first, request second row
f1_Score=0.515  precision=0.406 recall=0.704    roc_auc=0.723   pr_auc=0.345    
f1_Score=0.519  precision=0.411 recall=0.703    roc_auc=0.713   pr_auc=0.353    


e:\miniconda\envs\mlops\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


,feature,test,statistic,p_value,drift_detected,note


Reference first, request second row
f1_Score=0.515  precision=0.406 recall=0.704    roc_auc=0.723   pr_auc=0.345    
f1_Score=0.482  precision=0.376 recall=0.672    roc_auc=0.699   pr_auc=0.317    


,feature,test,statistic,p_value,drift_detected,note


Reference first, request second row
f1_Score=0.515  precision=0.406 recall=0.704    roc_auc=0.723   pr_auc=0.345    
f1_Score=0.487  precision=0.382 recall=0.671    roc_auc=0.705   pr_auc=0.321    


In [391]:
# сохраняем
req.to_csv(Path.cwd().parent / "data" / "interim" / "synt_3.csv")
datadrifted_data = req.copy()
del req